<a href="https://colab.research.google.com/github/delsucflorian/Oncolake_TorchProtein/blob/main/notebooks/03_prostt5_embeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

!nvidia-smi

Sat Sep 12 16:23:51 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import os
from google.colab import drive
drive.mount('/content/drive')
!pip install transformers sentencepiece -q
!pip install biopython -q

if not os.path.exists("/content/data/alphafold"):

  !mkdir -p /content/data
  !cp -r "/content/drive/MyDrive/oncolake_torchprotein/data/alphafold" /content/data/
else :
  print("Rien a installer coté drive ")
if not os.path.exists("/content/foldseek/bin/foldseek"):
  !wget https://mmseqs.com/foldseek/foldseek-linux-avx2.tar.gz -q
  !tar xvfz foldseek-linux-avx2.tar.gz > /dev/null
  !chmod +x foldseek/bin/foldseek
  !foldseek/bin/foldseek version
else:
  print("Rien a installer coté foldseek")

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 47.3 MB/s eta 0:00:00
463739e0014a1549a527de589102cde98f802f37


In [3]:
import transformers
print(f"Transformers : {transformers.__version__}")

Transformers : 5.16.1


In [4]:
!mkdir -p /content/foldseek_output
!foldseek/bin/foldseek createdb /content/data/alphafold /content/foldseek_output/db


createdb /content/data/alphafold /content/foldseek_output/db 

MMseqs Version:         	463739e0014a1549a527de589102cde98f802f37
Use GPU                 	0
Path to ProstT5         	
Chain name mode         	0
Model name mode         	0
Write mapping file      	0
Write Foldcomp          	0
Mask b-factor threshold 	0
Coord store mode        	2
Save residue indices    	false
Write lookup file       	1
Input format            	0
Input compression format	0
File Inclusion Regex    	.*
File Exclusion Regex    	^$
Threads                 	2
Verbosity               	3

Output file: /content/foldseek_output/db
[=================================================================] 100.00% 410 10s 872ms
Time for merging to db_ss: 0h 0m 0s 0ms
Time for merging to db_h: 0h 0m 0s 0ms
Time for merging to db_ca: 0h 0m 0s 3ms
Time for merging to db: 0h 0m 0s 2ms
Ignore 0 out of 410.
Too short: 0, incorrect: 0, not proteins: 0.
Time for processing: 0h 0m 10s 919ms


In [5]:
!ls -la /content/foldseek_output/

total 2152
drwxr-xr-x 2 root root    4096 Sep 12 16:25 .
drwxr-xr-x 1 root root    4096 Sep 12 16:25 ..
-rw-r--r-- 1 root root  262983 Sep 12 16:25 db
-rw-r--r-- 1 root root 1576258 Sep 12 16:25 db_ca
-rw-r--r-- 1 root root       4 Sep 12 16:25 db_ca.dbtype
-rw-r--r-- 1 root root    6541 Sep 12 16:25 db_ca.index
-rw-r--r-- 1 root root       4 Sep 12 16:25 db.dbtype
-rw-r--r-- 1 root root   16705 Sep 12 16:25 db_h
-rw-r--r-- 1 root root       4 Sep 12 16:25 db_h.dbtype
-rw-r--r-- 1 root root    4938 Sep 12 16:25 db_h.index
-rw-r--r-- 1 root root    5932 Sep 12 16:25 db.index
-rw-r--r-- 1 root root    5934 Sep 12 16:25 db.lookup
-rw-r--r-- 1 root root    4393 Sep 12 16:25 db.source
-rw-r--r-- 1 root root  262983 Sep 12 16:25 db_ss
-rw-r--r-- 1 root root       4 Sep 12 16:25 db_ss.dbtype
-rw-r--r-- 1 root root    5932 Sep 12 16:25 db_ss.index


In [6]:
!foldseek/bin/foldseek lndb /content/foldseek_output/db_h /content/foldseek_output/db_ss_h
!foldseek/bin/foldseek convert2fasta /content/foldseek_output/db_ss /content/foldseek_output/3di_sequences.fasta

lndb /content/foldseek_output/db_h /content/foldseek_output/db_ss_h 

MMseqs Version:	463739e0014a1549a527de589102cde98f802f37
Verbosity	3

Time for processing: 0h 0m 0s 0ms
convert2fasta /content/foldseek_output/db_ss /content/foldseek_output/3di_sequences.fasta 

MMseqs Version:	463739e0014a1549a527de589102cde98f802f37
Use header DB	false
Verbosity    	3

Start writing file to /content/foldseek_output/3di_sequences.fasta
Time for processing: 0h 0m 0s 0ms


In [7]:
!head -6 /content/foldseek_output/3di_sequences.fasta
!grep -c "^>" /content/foldseek_output/3di_sequences.fasta

>A0A2R8Y7D0 Ubiquitin domain-containing protein TINCR
DDDDDDDDDPDDWAFEWEQELVVSDTDTDIHGQPDFLLVVCVVCVVVVHPQPLWFKDFLLDTDDRRDGCNNVVNHHHGYIYTDNDSVSVVVCSVVVVVVVVVVVVVVVVVVPPPDPDPDD
>A1YPR0 Zinc finger and BTB domain-containing protein 7C
DPPPPPDPDDDDDPCPVQVVLQVQVVCQVVLHPFQEWEDEPHDIGGHDLVLLLVQFPQSVVVVVPDPDDDHRDYFYDDQADPVLVVQSVCCSRNVKGWDDPVCLVRVLSVCVVRVRVVSVVVSVVVVDPDDDDDDDDDDDPPPPPPCDDPPDPPDDDDDDDDDDDDDDDDDDDDDDDDDDPPPPLPDDDDDDDDDDDDDDDDDDDDDDDDDDDDDDDDDIDTDIDHVVVVVVNVPDDDPDDDDDDDDDDDDDDDDDDDDDDDDDDDDDDDDDDDDDDDDPPDPGPPDPDDDDDDDDDDDDDDDDDPPCPVVVVDPDDDDDDDDDDDPPDDPLVVVQVVCPPPVVPPDDNDPPPPDPPVPQQDWDAAPPPRDTDGHPVVNVLVVCSVVVDFPDADPPPRDTHSDVVVVVLVCCVVVVDQPDADPVNRDGHSDPVVNQLVVCSVVVDFPDADPQAGDGHSDPVVSVVCVVVVVSVRPDPPPPDDDPVVVVVCVVDDPDDDDDDDDDPDDDDDDDDDDDDDDDDDDDDDDDDDDDDDDDDDDDDDDPVVVVVVVVVVVVVVVVVVVVVVVVVVVPDDDDDDDDDDDDDDDDDDDDDDDDDDDDDDDDDDDDDDDDDDDDDDD
>A2RRD8 Zinc finger protein 320
DPPPPDLDDLVNVQDDDDPVVVVVDDPVRVVVSVVSSVVSVVVSVVVVLVVVVVVVVPPDDDDDDDDDDDPDDPDDDDDDDPPPPPVVVNVVVVVVVVVVVVVVVVPPDDDDDDDP

In [8]:
from transformers import T5Tokenizer, T5EncoderModel
import torch


MODEL_NAME = "Rostlab/ProstT5"
print("Chargement du tokenizer...")
tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME, do_lower_case=False)
print("Chargement du modèle (peut prendre 1-2 min, ~2 GB à télécharger)...")
model = T5EncoderModel.from_pretrained(MODEL_NAME)
model = model.eval().to("cuda")
model = model.half()

print(f"\nModèle chargé sur : {next(model.parameters()).device}")
print(f"Nombre de paramètres : {sum(p.numel() for p in model.parameters())/1e6:.0f}M")

Chargement du tokenizer...


tokenizer_config.json:   0%|          | 0.00/2.60k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  238kB            

spiece.model: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/283 [00:00<?, ?B/s]

...

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 11.3GB            

model.safetensors: downloading bytes:           |  0.00B            


Modèle chargé sur : cuda:0
Nombre de paramètres : 1208M


In [9]:
from Bio import SeqIO

import pandas as pd
features_ref = pd.read_parquet('/content/drive/MyDrive/oncolake_torchprotein/data/features_baseline_ref.parquet')
accessions_404 = features_ref['accession'].tolist()
print(f"Nombre d'accessions dans le parquet: {len(accessions_404)}")
FASTA_3DI = '/content/foldseek_output/3di_sequences.fasta'


sequences_3di = {}
for record in SeqIO.parse(FASTA_3DI, "fasta"):
    accession = record.id
    sequence = str(record.seq)
    if accession in accessions_404:
      sequences_3di[accession] = sequence
print(f"Nombre de séquences chargées : {len(sequences_3di)}")

lengths = [len(s) for s in sequences_3di.values()]
print(f"Longueur min : {min(lengths)}")
print(f"Longueur max : {max(lengths)}")
print(f"Longueur médiane : {sorted(lengths)[len(lengths)//2]}")

for i, (acc, seq) in enumerate(list(sequences_3di.items())[:3]):
    print(f"\n{acc} (longueur {len(seq)})")
    print(f"  {seq[:80]}...")
missing = set(accessions_404) - set(sequences_3di.keys())
print(f"Accessions manquantes ({len(missing)}) : {missing}")

Nombre d'accessions dans le parquet: 404
Nombre de séquences chargées : 404
Longueur min : 68
Longueur max : 2696
Longueur médiane : 500

A0A2R8Y7D0 (longueur 120)
  DDDDDDDDDPDDWAFEWEQELVVSDTDTDIHGQPDFLLVVCVVCVVVVHPQPLWFKDFLLDTDDRRDGCNNVVNHHHGYI...

A1YPR0 (longueur 619)
  DPPPPPDPDDDDDPCPVQVVLQVQVVCQVVLHPFQEWEDEPHDIGGHDLVLLLVQFPQSVVVVVPDPDDDHRDYFYDDQA...

A2RRD8 (longueur 509)
  DPPPPDLDDLVNVQDDDDPVVVVVDDPVRVVVSVVSSVVSVVVSVVVVLVVVVVVVVPPDDDDDDDDDDDPDDPDDDDDD...
Accessions manquantes (0) : set()


In [10]:
from tqdm import tqdm
import numpy as np
import torch

def get_embedding(seq_3di, model, tokenizer, device='cuda', max_length=1500):
    """
    Retourne un vecteur numpy (1024,) qui représente la protéine.
    """
    seq = seq_3di
    if len(seq) > max_length:
        seq = seq[:max_length]
    seq = '<fold2AA> ' + ' '.join(seq.lower())
    tokens = tokenizer(seq, return_tensors='pt', padding=True, truncation=True).to(device)
    with torch.no_grad():
        outputs = model(input_ids=tokens['input_ids'], attention_mask=tokens['attention_mask'])
    outputs = outputs.last_hidden_state
    final_output = torch.mean(outputs, dim=1)
    # 8. Retourner un numpy array
    return final_output.cpu().numpy().squeeze()


In [11]:
test_acc = list(sequences_3di.keys())[0]
test_seq = sequences_3di[test_acc]
emb = get_embedding(test_seq, model, tokenizer)
print(f"Test sur {test_acc}, shape : {emb.shape}")

Test sur A0A2R8Y7D0, shape : (1024,)


In [12]:
embeddings_dict = {}
for accession, seq in tqdm(sequences_3di.items(), desc="Embeddings"):
  embeddings_dict[accession] = get_embedding(seq, model, tokenizer)



Embeddings:   0%|          | 0/404 [00:00<?, ?it/s]

Embeddings:   0%|          | 2/404 [00:00<01:01,  6.56it/s]

Embeddings:   1%|          | 3/404 [00:00<01:04,  6.21it/s]

Embeddings:   1%|          | 5/404 [00:01<02:21,  2.82it/s]

Embeddings:   1%|▏         | 6/404 [00:01<02:21,  2.81it/s]

Embeddings:   2%|▏         | 8/404 [00:02<01:30,  4.37it/s]

Embeddings:   2%|▏         | 9/404 [00:02<01:21,  4.87it/s]

Embeddings:   2%|▏         | 10/404 [00:02<02:06,  3.11it/s]

Embeddings:   3%|▎         | 11/404 [00:03<02:32,  2.57it/s]

Embeddings:   3%|▎         | 12/404 [00:03<02:35,  2.52it/s]

Embeddings:   3%|▎         | 13/404 [00:04<02:19,  2.80it/s]

Embeddings:   3%|▎         | 14/404 [00:04<02:02,  3.17it/s]

Embeddings:   4%|▎         | 15/404 [00:04<01:53,  3.44it/s]

Embeddings:   4%|▍         | 16/404 [00:04<01:38,  3.94it/s]

Embeddings:   4%|▍         | 17/404 [00:05<01:47,  3.59it/s]

Embeddings:   4%|▍         | 18/404 [00:05<01:52,  3.43it/s]

Embeddings:   5%|▍   

In [14]:
first_acc = list(embeddings_dict.keys())[0]
last_acc = list(embeddings_dict.keys())[-1]
first_emb = embeddings_dict[first_acc]
last_emb = embeddings_dict[last_acc]

print(f"Premier ({first_acc}) : shape {first_emb.shape}, mean {first_emb.mean():.4f}")
print(f"Dernier ({last_acc}) : shape {last_emb.shape}, mean {last_emb.mean():.4f}")
print(f"Identiques ? {np.allclose(first_emb, last_emb)}")

Premier (A0A2R8Y7D0) : shape (1024,), mean 0.0008
Dernier (Q9Y6E7) : shape (1024,), mean -0.0008
Identiques ? False


In [15]:
import numpy as np
import json

accessions_ordered = list(embeddings_dict.keys())
embeddings_matrix = np.stack([embeddings_dict[acc] for acc in accessions_ordered])
assert embeddings_matrix.shape == (404, 1024), f"Shape inattendue : {embeddings_matrix.shape}"
assert len(accessions_ordered) == 404

print(f"Matrice : shape {embeddings_matrix.shape}, dtype {embeddings_matrix.dtype}")

results_dir = '/content/drive/MyDrive/oncolake_torchprotein/embeddings'
!mkdir -p "{results_dir}"

# Matrice numpy
np.save(f"{results_dir}/prostt5_embeddings.npy", embeddings_matrix)
with open(f"{results_dir}/prostt5_accessions.json", 'w') as f:
    json.dump(accessions_ordered, f, indent=2)

print(f"Sauvegardés dans {results_dir}")
!ls -la "{results_dir}"

Matrice : shape (404, 1024), dtype float16
Sauvegardés dans /content/drive/MyDrive/oncolake_torchprotein/embeddings
total 822
drwx------ 2 root root   4096 Sep 12 16:34 .
drwx------ 6 root root   4096 Sep 12 16:34 ..
-rw------- 1 root root   4854 Sep 12 16:34 prostt5_accessions.json
-rw------- 1 root root 827520 Sep 12 16:34 prostt5_embeddings.npy


## Summary

This notebook produces structural embeddings for the 404 proteins of the
OncoLake dataset, to be compared against the handcrafted feature baseline
(F1 macro 0.47 ± 0.04) established in notebook 02.

- Q9UGM3 (DMBT1) was absent from the local `.cif` snapshot but present in the
  reference feature parquet. Re-downloaded from AlphaFold DB (v6, whereas the
  original OncoLake pipeline used v4) to ensure full 404-protein coverage.
  Expected impact negligible: same AlphaFold2 model, incremental retraining only.

**Result**
- Embedding matrix of shape `(404, 1024)` in float16 precision.

**Saved artifacts**
- `embeddings/prostt5_embeddings.npy` — the embedding matrix
- `embeddings/prostt5_accessions.json` — ordered list of accessions (row `i`
  of the matrix corresponds to accession `i` in this file)

**Next**

Notebook 04 will evaluate this embedding-based representation against the
OncoLake handcrafted baseline using the pre-declared protocol from notebook 02
(GroupKFold on MMseqs2 clusters, F1 macro, 5 seeds × 5 folds, Wilcoxon paired
test for comparison).